# Lab: the Nobel Prize, from bronze to gold

In the first notebook we explored the raw JSON files. Now we turn them into a small data warehouse, one layer at a time:

| Layer | What it holds | Who uses it |
|---|---|---|
| 🥉 **Bronze** | The data exactly as the API sent it | Nobody directly: it is our safety copy |
| 🥈 **Silver** | Clean, typed tables: one row per laureate, one row per award | Analysts and engineers |
| 🥇 **Gold** | Tables shaped to answer questions | Reports and dashboards |

At the end we answer three questions:

1. Which countries have the most laureates?
2. How old are laureates when they win?
3. How has the share of women changed?

## Part 0: Setup

This time we connect to a **file**. The tables we create are saved in it, so they are still there the next time we open the notebook.

In [1]:
import duckdb

con = duckdb.connect('../data/nobel.duckdb')

Each layer gets its own **schema**. A schema works like a folder for tables: `bronze.nobel_pages` is the table `nobel_pages` inside the folder `bronze`.

In [2]:
con.sql("CREATE SCHEMA IF NOT EXISTS bronze")
con.sql("CREATE SCHEMA IF NOT EXISTS silver")
con.sql("CREATE SCHEMA IF NOT EXISTS gold")

CountBIGINT


---
## Part 1: Bronze

The rule of bronze: **change nothing**. Each JSON file (one page of the API) becomes one row. We only add two columns of our own:

- `filename`: which file the row came from
- `loaded_at`: when we loaded it

In [3]:
con.sql("""
    CREATE OR REPLACE TABLE bronze.nobel_pages AS
    SELECT *, now() AS loaded_at
    FROM read_json('../data/bronze/*.json', filename = true)
""")

con.sql("SELECT filename, len(laureates) AS laureates_in_page, loaded_at FROM bronze.nobel_pages").df()

,filename,laureates_in_page,loaded_at
0,../data/bronze/laureates_offset0000.json,100,2026-11-15 11:24:05.098853+01:00
1,../data/bronze/laureates_offset0100.json,100,2026-11-15 11:24:05.098853+01:00
2,../data/bronze/laureates_offset0200.json,100,2026-11-15 11:24:05.098853+01:00
3,../data/bronze/laureates_offset0300.json,100,2026-11-15 11:24:05.098853+01:00
4,../data/bronze/laureates_offset0400.json,100,2026-11-15 11:24:05.098853+01:00
5,../data/bronze/laureates_offset0500.json,100,2026-11-15 11:24:05.098853+01:00
6,../data/bronze/laureates_offset0600.json,100,2026-11-15 11:24:05.098853+01:00
7,../data/bronze/laureates_offset0700.json,100,2026-11-15 11:24:05.098853+01:00
8,../data/bronze/laureates_offset0800.json,100,2026-11-15 11:24:05.098853+01:00
9,../data/bronze/laureates_offset0900.json,100,2026-11-15 11:24:05.098853+01:00


### Did we download everything?

Every page says how many laureates the API has in total (`meta.count`). That number should be equal to the number of laureates we actually have in our pages.

In [4]:
con.sql("""
    SELECT max(meta.count)                         AS api_says,
           CAST(sum(len(laureates)) AS INTEGER)    AS we_have
    FROM bronze.nobel_pages
""").df()

,api_says,we_have
0,1028,1028


Both numbers are the same, so nothing is missing.

> **Think about it:** why keep the pages exactly as the API sent them, instead of saving a cleaned table straight away?
>
> Because cleaning rules change, and sometimes they have bugs. With the original data stored, we can rebuild silver and gold at any time, without downloading again.

---
## Part 2: Silver

Silver is where the cleaning happens. We build two tables:

- `silver.laureates`: one row per laureate
- `silver.awards`: one row per prize won

### Step 1: open the list

In bronze, 100 laureates are packed inside one row. `unnest` opens the list: one row per laureate, one column per field. This is the same thing we did in the first notebook.

We save it as a **view**: a saved query with a name. It stores no data, it just reads bronze every time we use it.

In [5]:
con.sql("""
    CREATE OR REPLACE VIEW silver.laureates_flat AS
    SELECT unnest(laureates, max_depth := 2)
    FROM bronze.nobel_pages
""")

con.sql("SELECT id, knownName, orgName, gender, birth FROM silver.laureates_flat LIMIT 5").df()

,id,knownName,orgName,gender,birth
0,745,"{'en': 'A. Michael Spence', 'se': 'A. Michael ...",<NA>,male,"{'date': '1943-11-07', 'year': '1943', 'place'..."
1,102,"{'en': 'Aage N. Bohr', 'se': 'Aage N. Bohr', '...",<NA>,male,"{'date': '1922-06-19', 'year': '1922', 'place'..."
2,779,"{'en': 'Aaron Ciechanover', 'se': 'Aaron Ciech...",<NA>,male,"{'date': '1947-10-01', 'year': '1947', 'place'..."
3,259,"{'en': 'Aaron Klug', 'se': 'Aaron Klug', 'no':...",<NA>,male,"{'date': '1926-08-11', 'year': '1926', 'place'..."
4,1004,"{'en': 'Abdulrazak Gurnah', 'se': 'Abdulrazak ...",<NA>,male,"{'date': '1948-00-00', 'year': '1948', 'place'..."


### Step 2: the laureates table

Now we choose the columns we want and fix the problems we found while exploring:

| Problem in the raw data | What we do |
|---|---|
| `id` is text | `CAST` it to a number |
| Names come in three languages | Keep only English (`.en`) |
| People have `knownName`, organisations have `orgName` | `coalesce` takes the first one that is not empty |
| People and organisations are mixed | Add a `laureate_type` column to tell them apart |
| Some birth dates are incomplete (`1948-00-00`) | `try_cast` returns an empty value instead of stopping with an error |

In [6]:
con.sql("""
    CREATE OR REPLACE TABLE silver.laureates AS
    SELECT
        CAST(id AS INTEGER)                        AS laureate_id,
        CASE WHEN orgName IS NOT NULL
             THEN 'organisation' ELSE 'person' END AS laureate_type,
        coalesce(knownName.en, orgName.en)         AS name,
        gender,
        coalesce(birth.place.countryNow.en,
                 founded.place.countryNow.en)      AS country,
        birth.date                                 AS birth_date_raw,
        try_cast(birth.date AS DATE)               AS birth_date,
        try_cast(left(birth.date, 4) AS INTEGER)   AS birth_year
    FROM silver.laureates_flat
""")

con.sql("SELECT * FROM silver.laureates LIMIT 5").df()

,laureate_id,laureate_type,name,gender,country,birth_date_raw,birth_date,birth_year
0,745,person,A. Michael Spence,male,USA,1943-11-07,1943-11-07,1943
1,102,person,Aage N. Bohr,male,Denmark,1922-06-19,1922-06-19,1922
2,779,person,Aaron Ciechanover,male,Israel,1947-10-01,1947-10-01,1947
3,259,person,Aaron Klug,male,Lithuania,1926-08-11,1926-08-11,1926
4,1004,person,Abdulrazak Gurnah,male,NaN,1948-00-00,NaT,1948


Look at what happened to the incomplete dates. We could not build a real date, so `birth_date` is empty, but we did not lose the year: it is the first 4 characters of the text.

We also kept the original text in `birth_date_raw`, so anyone can see what the source really said.

In [7]:
con.sql("""
    SELECT name, birth_date_raw, birth_date, birth_year
    FROM silver.laureates
    WHERE birth_date_raw LIKE '%-00-00'
    LIMIT 5
""").df()

,name,birth_date_raw,birth_date,birth_year
0,Abdulrazak Gurnah,1948-00-00,NaT,1948
1,Albert Lutuli,1898-00-00,NaT,1898
2,Aleksey Yekimov,1945-00-00,NaT,1945
3,Ardem Patapoutian,1967-00-00,NaT,1967
4,David Baker,1962-00-00,NaT,1962


### Check: one row per laureate

A good habit after building a table: count the rows. We expect 1028 rows, 1028 different ids, and nobody without a name.

In [8]:
con.sql("""
    SELECT count(*)                    AS total_rows,
           count(DISTINCT laureate_id) AS different_ids,
           count(name)                 AS with_name
    FROM silver.laureates
""").df()

,total_rows,different_ids,with_name
0,1028,1028,1028


> **Think about it:** for `country` we used `countryNow`, the country that exists **today** at the place of birth. Someone born in Prussia is counted under Germany or Poland.
>
> This is a good choice to compare countries today, and a bad one for history questions. It is a decision, not a fact, so it should be written down.

### Step 3: the awards table

A laureate can win more than once (Marie Curie: 1903 and 1911). So the prizes are a list inside each laureate, and they need their own table: one row per prize won.

We `unnest` again, this time the `nobelPrizes` list. We keep the laureate's `id` next to each prize, so we know who won it.

Many prizes are shared. The `portion` says how much of the prize each laureate got, but it is written as text: `'1/3'`. We turn it into a number so we can add it up later.

In [9]:
con.sql("""
    CREATE OR REPLACE TABLE silver.awards AS
    SELECT
        CAST(id AS INTEGER)              AS laureate_id,
        CAST(prize.awardYear AS INTEGER) AS award_year,
        prize.category.en                AS category,
        prize.portion                    AS portion_raw,
        CASE prize.portion
            WHEN '1'   THEN 1.0
            WHEN '1/2' THEN 1.0 / 2
            WHEN '1/3' THEN 1.0 / 3
            WHEN '1/4' THEN 1.0 / 4
        END                              AS portion,
        prize.prizeAmount                AS prize_amount_sek
    FROM (SELECT id, unnest(nobelPrizes) AS prize FROM silver.laureates_flat)
""")

con.sql("SELECT * FROM silver.awards ORDER BY award_year, category LIMIT 8").df()

,laureate_id,award_year,category,portion_raw,portion,prize_amount_sek
0,160,1901,Chemistry,1,1.0,150782
1,569,1901,Literature,1,1.0,150782
2,463,1901,Peace,1/2,0.5,150782
3,462,1901,Peace,1/2,0.5,150782
4,1,1901,Physics,1,1.0,150782
5,293,1901,Physiology or Medicine,1,1.0,150782
6,161,1902,Chemistry,1,1.0,141847
7,571,1902,Literature,1,1.0,141847


### Check: the portions must add up

Each prize (one year, one category) is split among its laureates, so the portions of one prize must add up to exactly 1.

This is a simple test that catches many mistakes: a wrong portion, a missing laureate, or a laureate counted twice. The query below lists the prizes where the total is **not** 1.

In [10]:
con.sql("""
    SELECT award_year, category,
           count(*)               AS laureates,
           round(sum(portion), 6) AS total_portion
    FROM silver.awards
    GROUP BY award_year, category
    HAVING round(sum(portion), 6) <> 1
""").df()

,award_year,category,laureates,total_portion


The result is empty: every prize adds up to 1. Silver is ready.

---
## Part 3: Gold

Gold tables are built for questions. We use a classic shape, the **star schema**:

- a **dimension** table describes *who*: `gold.dim_laureate`
- a **fact** table records *what happened*: `gold.fct_award`, one row for each laureate winning a prize

They are linked by `laureate_id`.

### The dimension: who are the laureates?

Organisations have no gender. Instead of leaving it empty, we write `'organisation'`, so reports do not show a mysterious blank group.

In [11]:
con.sql("""
    CREATE OR REPLACE TABLE gold.dim_laureate AS
    SELECT laureate_id,
           name,
           laureate_type,
           coalesce(gender, 'organisation') AS gender,
           country,
           birth_year
    FROM silver.laureates
""")

con.sql("SELECT gender, count(*) AS total FROM gold.dim_laureate GROUP BY gender").df()

,gender,total
0,female,69
1,male,931
2,organisation,28


### The fact: who won what, and when?

We add two columns that make the questions easy to answer:

- `decade`: 1903 becomes 1900. `//` is a division that throws away the remainder, so `1903 // 10` is `190`, and `190 * 10` is `1900`.
- `age_at_award`: the award year minus the birth year. Organisations have no birth year, so their age stays empty.

In [12]:
con.sql("""
    CREATE OR REPLACE TABLE gold.fct_award AS
    SELECT a.laureate_id,
           a.award_year,
           (a.award_year // 10) * 10    AS decade,
           a.category,
           a.portion,
           a.prize_amount_sek,
           a.award_year - l.birth_year  AS age_at_award
    FROM silver.awards a
    JOIN gold.dim_laureate l USING (laureate_id)
""")

con.sql("SELECT * FROM gold.fct_award LIMIT 5").df()

,laureate_id,award_year,decade,category,portion,prize_amount_sek,age_at_award
0,745,2001,2000,Economic Sciences,0.333333,10000000,58
1,102,1975,1970,Physics,0.333333,630000,53
2,779,2004,2000,Chemistry,0.333333,10000000,57
3,259,1982,1980,Chemistry,1.000000,1150000,56
4,1004,2021,2020,Literature,1.000000,10000000,73


### Question 1: which countries have the most laureates?

There are two ways to count:

- **awards**: how many times someone from that country won (the number of rows)
- **prizes**: the sum of the portions. Three laureates sharing one prize count as one prize in total.

In [13]:
con.sql("""
    SELECT l.country,
           count(*)                 AS awards,
           round(sum(f.portion), 1) AS prizes
    FROM gold.fct_award f
    JOIN gold.dim_laureate l USING (laureate_id)
    GROUP BY l.country
    ORDER BY awards DESC
    LIMIT 10
""").df()

,country,awards,prizes
0,USA,309,162.8
1,United Kingdom,97,56.2
2,Germany,86,52.3
3,France,66,45.0
4,Japan,32,15.8
5,Sweden,30,21.4
6,Russia,30,19.3
7,Poland,29,23.2
8,Switzerland,27,18.9
9,Canada,24,14.8


### Question 2: how old are laureates when they win?

In [14]:
con.sql("""
    SELECT category,
           round(avg(age_at_award), 1) AS average_age,
           min(age_at_award)           AS youngest,
           max(age_at_award)           AS oldest
    FROM gold.fct_award
    GROUP BY category
    ORDER BY average_age
""").df()

,category,average_age,youngest,oldest
0,Physics,57.8,25,96
1,Physiology or Medicine,59.0,32,87
2,Chemistry,59.5,35,97
3,Peace,61.0,17,87
4,Literature,65.0,42,88
5,Economic Sciences,67.2,47,90


> **Think about it:** adding up `portion` gives a number of prizes, which makes sense. What would `sum(age_at_award)` mean?
>
> Nothing. Ages can be averaged, but not added up. Not every number in a table can be summed.

### Question 3: how has the share of women changed?

Only people, since organisations have no gender. For each decade we count everyone, count the women, and divide.

In [15]:
con.sql("""
    SELECT f.decade,
           count(*)                                    AS people,
           count(*) FILTER (WHERE l.gender = 'female') AS women,
           round(100.0 * count(*) FILTER (WHERE l.gender = 'female') / count(*), 1) AS percent_women
    FROM gold.fct_award f
    JOIN gold.dim_laureate l USING (laureate_id)
    WHERE l.laureate_type = 'person'
    GROUP BY f.decade
    ORDER BY f.decade
""").df()

,decade,people,women,percent_women
0,1900,56,3,5.4
1,1910,38,1,2.6
2,1920,54,2,3.7
3,1930,55,3,5.5
4,1940,40,3,7.5
5,1950,71,0,0.0
6,1960,75,3,4.0
7,1970,103,4,3.9
8,1980,94,4,4.3
9,1990,101,7,6.9


---
## Part 4: Wrap-up

Here is everything we built. Three schemas, one per layer, all inside the same database file:

In [16]:
con.sql("SELECT schema, name FROM (SHOW ALL TABLES) ORDER BY schema, name").df()

,schema,name
0,bronze,nobel_pages
1,gold,dim_laureate
2,gold,fct_award
3,silver,awards
4,silver,laureates
5,silver,laureates_flat


What each layer did:

- **Bronze** stored the pages untouched, with the file name and the load time.
- **Silver** opened the nested lists, gave columns clear names and real types, handled incomplete dates, and separated laureates from awards. We checked it twice: one row per laureate, and portions that add up to 1.
- **Gold** arranged the data as a dimension and a fact, and answered the three questions.

Every table was created with `CREATE OR REPLACE`, so you can run the whole notebook again at any time and get the same result.

When you are done, close the connection. This saves everything and releases the database file.

In [17]:
con.close()

### Try it yourself

Run the first cell again to reconnect, then:

1. Which laureates won **more than once**? Use `gold.fct_award`, group by `laureate_id`, and keep the groups `HAVING count(*) > 1`.
2. What is the average prize amount per decade? The amounts are in Swedish kronor of the time: why is comparing 1901 with 2020 misleading?
3. Rebuild `country` using the country **at the time of birth** (`birth.place.country.en`) instead of `countryNow`. How does the top 10 change?